# Load SmolLM2 & Compute Embedding Dot Products (standard Attention QK^T)

In [8]:
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained("HuggingFaceTB/SmolLM2-135M")
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1604.94it/s]


Index :

get_input_embeddings() — returns the embedding layer: a huge lookup table that maps token-ID → vector.

.weight — the actual matrix inside specific layer.

.detach() — "cut the gradient." PyTorch tracks every tensor for backpropagation. We only want numbers, not gradients, so we detach

.numpy() — convert the PyTorch tensor into a plain numpy array.

In [9]:
# grab emebedding matrix W_E
W_E = model.get_input_embeddings().weight.float().detach().numpy()

# ponder : numerical precision demand float d-type.

# dimenssions ~(50000, 576) / rounded-off output(vocab_size, dimensions)
print("Embeddings matrix shape:", W_E.shape)

# ponder : Encountering trusted measurement of model vocabulary

Embeddings matrix shape: (49152, 576)


# Embedding / Indexing 

The embedding layer is index-based retrieval that returns learned points; the dot product in attention then interrogates the geometry those points form.

In [12]:
# Dict > raw tokens > Lookup operation/ dict lookup

words = ["king", "queen", "table", "computer"]
ids = tokenizer.convert_tokens_to_ids(words)
print("Token IDs:", dict(zip(words, ids)))

vectors = W_E[ids] # shape (4, 576) - single row = word.

# ponder:
#  W_E is index-based retrieval (mechanics). But the 576 dims are arbitrary coordinates
#  training makes the GEOMETRY meaningful, and attention dot products read out that geometry.

Token IDs: {'king': 644, 'queen': 0, 'table': 6413, 'computer': 20528}


High dot = aligned direction + strong magnitude = related tokens

Low/negative dot = orthogonal/opposed = unrelated tokens

(attention uses RAW dot product, NOT cosine similarity, because magnitude is informative too)

In [ ]:
# compute dot-products amoung index-retrieved tokens/ids / semantic relation
for i, w1 in enumerate(words):
    for j, w2 in enumerate(words):
        dot = float(np.dot(vectors[i], vectors[j]))

        print(f"{w1:10s} . {w2:10s} = {dot:8.2f}")

# ponder:  A.B = |A||b|cos(0) similarity is dot-product(closeness of tokens) 
# here (0) is the angle b/w vectors.

king       . king       =    10.21
king       . queen      =    -0.20
king       . table      =     2.51
king       . computer   =     2.71
queen      . king       =    -0.20
queen      . queen      =     8.87
queen      . table      =    -0.64
queen      . computer   =    -0.75
table      . king       =     2.51
table      . queen      =    -0.64
table      . table      =     8.42
table      . computer   =     2.81
computer   . king       =     2.71
computer   . queen      =    -0.75
computer   . table      =     2.81
computer   . computer   =     7.08


Observations to diagnosis : 

here dot product isnt tokens (king) . (queen)

rather (king) . (end-of-text / special token) which tampers the deviation.

# Diagnosis


In [ ]:
# testing Singular Value Decomposion to prove anisotrophy (directional dependence) 

# using formula : A = U . S . Vt

U, S, Vt = np.linalg.svd(W_E, full_matrices=False)
print(S[0] / S.sum()) # frac of energy in top direction

# Dependent relevance / anisotrophy = (4.2%)

0.042252757


### Hypothetical Conclusion : 
 we can trouble shoot context anomalies with such method or test a fedility of models in terms of tokenization failures.

In [ ]:
# 
print(tokenizer.convert_ids_to_tokens(ids)) # OOV (out - of - vocab) error.

['king', '<|endoftext|>', 'table', 'computer']


# Solution

In [24]:
#  check decode(encode(w)) == w automatically drops queen and keeps only real tokens.

candidates = ["king", "queen", "table", "computer", "cat", "dog", "apple", "water", "city"]
ids = tokenizer.convert_tokens_to_ids(candidates)
tokens = tokenizer.convert_ids_to_tokens(ids)
good = [w for w, t in zip(candidates, tokens) if t == w]

print("Verified single-token words:", good)
assert len(good) >=3

Verified single-token words: ['king', 'table', 'computer', 'cat', 'dog', 'apple', 'water', 'city']


In [ ]:
# measure direction of cosines / similarity
def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a)) * (np.linalg.norm(b))

words = good
vecs = W_E[tokenizer.convert_tokens_to_ids(words)]
for i, w1 in enumerate(words):
    row = "  ".join(f"{cos_sim(vecs[i], vecs[j]):6.3f}" for j in range(len(words)))
    print(f"{w1:10s} {row}")

king       10.205   2.276   2.259   2.231   2.799   4.686   2.550   2.016
table       2.758   8.422   2.579   1.925   3.227   3.985   2.348   2.514
computer    3.256   3.068   7.080   2.797   3.933   5.715   3.463   3.117
cat         3.313   2.359   2.881   6.872   4.701   5.301   3.044   2.806
dog         3.428   3.261   3.342   3.877   8.332   5.855   3.386   2.947
apple       4.198   2.946   3.552   3.198   4.283  11.391   3.898   3.135
water       3.048   2.316   2.872   2.450   3.304   5.200   8.539   2.239
city        3.077   3.166   3.300   2.883   3.671   5.339   2.859   6.688


In [ ]:
# Quantified anisotrophy 

U, S, Vt = np.linalg.svd(W_E, full_matrices=False)
uniform = 1 / S.size
print(f"Top-1 singular share: {S[0]/S.sum():.4f}  (isotropic would be {uniform:.4f})")
print(f"Concentration factor: {(S[0]/S.sum())/uniform:.1f}x")